# Ñ · las etapas N4 (y N2, N3) en las dos T4 de Kaggle

Corre las suites de `motor-exp/pruebas/` que miden la GPU de verdad (`--gpu`), más dos de CPU que en Linux
no se habían confirmado tras los arreglos de Windows (`exp-aot --rapida`, `exp-lengua-franca`).
El proyecto sale del dataset `n-motor-exp` (la rama `motor-exp` con N4.6–N4.11). Acelerador: **GPU T4 x2**.

Lo que importa queda en `/kaggle/working/etapas-t4-resultados.txt` y `.json`.

In [ ]:
# 1 · Node y el proyecto (del dataset)
import os, subprocess, shutil, re, glob, json, time
TRABAJO = '/kaggle/temp/n-etapas'; SALIDA = '/kaggle/working'
os.makedirs(TRABAJO, exist_ok=True)
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C {TRABAJO}', shell=True, check=True)
    os.environ['PATH'] = f'{TRABAJO}/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())
PROYECTO = f'{TRABAJO}/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if not candidatos: raise SystemExit('no encuentro el proyecto en /kaggle/input: añade el dataset n-motor-exp')
if os.path.exists(PROYECTO): shutil.rmtree(PROYECTO)
shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules')); print('proyecto copiado desde', candidatos[0])
os.chdir(PROYECTO)
subprocess.run('npm install koffi --no-audit --no-fund --loglevel=error', shell=True, check=True, timeout=900)
print(subprocess.run('node motor-exp/util/construir.js', shell=True, capture_output=True, text=True).stdout.strip())
print(subprocess.run('nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv', shell=True, capture_output=True, text=True).stdout.strip())
print(subprocess.run('gcc --version | head -1; g++ --version | head -1; python3 -c "import numpy; print(\'numpy\', numpy.__version__)"', shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# 2 · Las suites, una por una. Cada una: su última línea (el resumen) y sus ✗.
SUITES = [
    ('N2 · NEÑ en la GPU',            'exp-nen-gpu.js --gpu'),
    ('N3 · BLAS propio (núcleos)',     'nen-nucleos.js'),
    ('C1EÑ · jit = nativo = gpu',      'c1en.js --gpu'),
    ('N4.2 · real32 y real16',         'exp-precision.js --gpu'),
    ('N4.3 · GPU asíncrona',           'exp-asincrono.js --gpu'),
    ('N4.4 · paraleloBloques',         'exp-bloques.js --gpu'),
    ('N4.5 · tipos de máquina',        'exp-tipos.js --gpu'),
    ('N4.7 · hilos y atómicos',        'exp-hilos.js --gpu'),
    ('N4.8 · varias GPU (las dos T4)', 'exp-varias-gpu.js --gpu'),
    ('N4.9 · plantillas',              'exp-plantillas.js --gpu'),
    ('N4.11 · sinComprobar',           'exp-sin-comprobar.js --gpu'),
    ('N4.1 · sin runtime (CPU)',       'exp-aot.js --rapida'),
    ('N4.6 · lengua franca (CPU)',     'exp-lengua-franca.js'),
]
resultados = []
for nombre, orden in SUITES:
    t0 = time.time()
    try:
        r = subprocess.run(f'node motor-exp/pruebas/{orden}', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        salida, codigo = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        salida, codigo = (e.stdout or '') + (e.stderr or '') + '\n(se pasó de 30 minutos)', 'tiempo'
    lineas = [l for l in salida.splitlines() if l.strip()]
    resumen = lineas[-1].strip() if lineas else 'sin salida'
    fallos = [l.strip() for l in lineas if l.lstrip().startswith('✗')]
    resultados.append(dict(etapa=nombre, orden=orden, codigo=codigo, segundos=round(time.time() - t0), resumen=resumen, fallos=fallos))
    print(f'{"✓" if codigo == 0 else "✗"} {nombre:32} {resumen}  ({round(time.time() - t0)} s)')
    for f in fallos[:12]: print('      ' + f)
    with open(f'{SALIDA}/salida-{orden.split()[0]}.txt', 'w') as f: f.write(salida)
json.dump(resultados, open(f'{SALIDA}/etapas-t4-resultados.json', 'w'), ensure_ascii=False, indent=1)
with open(f'{SALIDA}/etapas-t4-resultados.txt', 'w') as f:
    for r in resultados: f.write(f'{"✓" if r["codigo"] == 0 else "✗"} {r["etapa"]}: {r["resumen"]}\n' + ''.join('    ' + x + '\n' for x in r['fallos']))
print('\nen verde:', sum(r['codigo'] == 0 for r in resultados), 'de', len(resultados))